# LCA Lasers4MaaS 



In [18]:
from pandas import DataFrame as df
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

## Workflow assembly

In [19]:
# Product-specific workflow configuration
PRODUCT_WORKFLOWS = {
    "Heat exchanger": {
        "material": "aluminium",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,
            "machining": False
        },
        "post_joining": {
            "chemical_etching": True,
            "conversion_coating": True,
            "grinding_polishing": True
        },
        "process_parameters": {
            "prototype" : {
                "surface_area_m2": {"min": 0.5, "max": 2.0},
                "part_weight_kg": {"min": 1.0, "max": 5.0},
                "joint_length_m": {"min": 5.0, "max": 15.0}
            }
            ,
            "production" : {
                "surface_area_m2": {"min": 1.0, "max": 5.0},
                "part_weight_kg": {"min": 5.0, "max": 20.0},
                "joint_length_m": {"min": 10.0, "max": 30.0}
            }
        }
    },
    
    "Fuel tank": {
        "material": "steel",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,  # Oxides/scale likely present
            "machining": True  # Fuel tanks require tight tolerances
        },
        "post_joining": {
            "pickling": True,
            "passivation": True,
            "grinding_polishing": True
        },
        "process_parameters": {
            "prototype" : {
                "surface_area_m2": {"min": 2.0, "max": 10.0},
                "part_weight_kg": {"min": 10.0, "max": 50.0},
                "joint_length_m": {"min": 2.0, "max": 2.2}
            },
            "production" : {
                "surface_area_m2": {"min": 5.0, "max": 20.0},
                "part_weight_kg": {"min": 50.0, "max": 200.0},
                "joint_length_m": {"min": 2.0, "max": 2.2}
            }
        }
    },
    
    "Heterogeneous pipe": {
        "material": "steel",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,  # Oxides/scale present on pipes
            "machining": False
        },
        "post_joining": {
            "pickling": True,
            "passivation": True,
            "grinding_polishing": True
        },
        "process_parameters": {
            "pipe_diameter_mm": {"min": 50, "max": 200},
            "pipe_length_m": {"min": 1.0, "max": 10.0},
            "joint_length_m": {"min": 0.5, "max": 3.0}
        }
    },
    
    "Freeze Dryers Shelf": {
        "material": "steel",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,
            "machining": False
        },
        "post_joining": {
            "pickling": True,
            "passivation": True,
            "grinding_polishing": False
        },
        "process_parameters": {
            "prototype" : {
                "surface_area_m2": {"min": 1.0, "max": 3.0},
                "part_weight_kg": {"min": 20.0, "max": 100.0},
                "joint_length_m": {"min": 5.0, "max": 20.0}
            },
            "production" : {
                "surface_area_m2": {"min": 3.0, "max": 10.0},
                "part_weight_kg": {"min": 100.0, "max": 300.0},
                "joint_length_m": {"min": 10.0, "max": 40.0}
            }
        }
    },
    
    "Battery cooling plate": {
        "material": "aluminium",
        "pre_joining": {
            "alkaline_degreasing": True,
            "mechanical_treatment": True,
            "machining": False
        },
        "post_joining": {
            "chemical_etching": True,
            "conversion_coating": False,
            "grinding_polishing": True
        },
        "process_parameters": {
            "prototype" : {
                "surface_area_m2": {"min": 0.1, "max": 0.5},
                "part_weight_kg": {"min": 1.26, "max": 14.38},
                "joint_length_m": {"min": 0.5, "max": 2.0}
            },
            "production" : {
                "surface_area_m2": {"min": 0.5, "max": 2.0},
                "part_weight_kg": {"min": 14.38, "max": 57.52},
                "joint_length_m": {"min": 2.0, "max": 8.0}
            }
        }
    }
}


In [20]:
import json
from pathlib import Path

RESULTS_DIR = Path("results")

# Load all JSON files found in the results folder
results_data = {}
for json_file in sorted(RESULTS_DIR.glob("*LCIA*.json")):
    with open(json_file, "r", encoding="utf-8") as f:
        results_data[json_file.stem] = json.load(f)
    print(f"Loaded: {json_file.name}")

# Identify the two result sets by file name (adjust keywords if needed)
chemical_results = next((v for k, v in results_data.items() if "chem" in k.lower()), {})
joining_results = next((v for k, v in results_data.items() if "joining" in k.lower() or "beam" in k.lower()), {})

PRE_STEPS = ["alkaline_degreasing", "mechanical_treatment", "machining"]
POST_STEPS = ["chemical_etching", "conversion_coating", "pickling", "passivation", "grinding_polishing"]


def get_step_result(source, step, material):
    """Find process records by step and material, grouped by scenario.

    Chemical rows use the bare step as Key (e.g. "pickling"), while mechanical
    rows append the material (e.g. "machining_steel"), so accept both forms.
    """
    if isinstance(source, list):
        valid_keys = {step, f"{step}_{material}"}
        matches = [
            row for row in source
            if isinstance(row, dict)
            and row.get("Key") in valid_keys
            and material.lower() in row.get("Process", "").lower()
        ]
        return {
            row.get("Scenario", "amount"): row
            for row in matches
        } or None

    if isinstance(source, dict):
        if step in source and isinstance(source[step], dict):
            if material in source[step]:
                return source[step][material]
        if material in source and isinstance(source[material], dict):
            return source[material].get(step)
        return source.get(step)

    return None


def assemble_product_results(product, config):
    material = config["material"]
    assembled = {"material": material, "pre_joining": {}, "joining": {}, "post_joining": {}}

    for stage, steps in (("pre_joining", PRE_STEPS), ("post_joining", POST_STEPS)):
        for step in steps:
            if config[stage].get(step):
                result = get_step_result(chemical_results, step, material)
                if result is None:
                    print(f"WARNING: no LCIA data for '{step}' ({material}) in {product} - plotted as 0")
                assembled[stage][step] = result

    # Match joining records by product name and group them by process and scenario.
    if isinstance(joining_results, list):
        for row in joining_results:
            if isinstance(row, dict) and row.get("Key") == product:
                process = row.get("Process", "Joining")
                scenario = row.get("Scenario", "amount")
                assembled["joining"].setdefault(process, {})[scenario] = row
    elif isinstance(joining_results, dict):
        assembled["joining"] = joining_results

    return assembled

assembled_results = {
    product: assemble_product_results(product, config)
    for product, config in PRODUCT_WORKFLOWS.items()
}

for product, res in assembled_results.items():
    print(f"\n{product} ({res['material']})")
    print("  Pre-joining :", list(res["pre_joining"].keys()))
    print("  Joining     :", list(res["joining"].keys()))
    print("  Post-joining:", list(res["post_joining"].keys()))

Loaded: Chemical_Pre_and_Post_processes_Mechanical_Pre_and_Post_processes_LCIA_results.json
Loaded: State_of_the_Art_Dynamic_Beam_Joining_LCIA_results.json

Heat exchanger (aluminium)
  Pre-joining : ['alkaline_degreasing', 'mechanical_treatment']
  Joining     : ['Heat exchanger joined with Dynamic Beam Joining', 'Heat exchanger joined with standard laser']
  Post-joining: ['chemical_etching', 'conversion_coating', 'grinding_polishing']

Fuel tank (steel)
  Pre-joining : ['alkaline_degreasing', 'mechanical_treatment', 'machining']
  Joining     : ['Fuel tank joined with Dynamic Beam Joining', 'Fuel tank joined with TIG']
  Post-joining: ['pickling', 'passivation', 'grinding_polishing']

Heterogeneous pipe (steel)
  Pre-joining : ['alkaline_degreasing', 'mechanical_treatment']
  Joining     : ['Heterogeneous pipe joined with Dynamic Beam Joining', 'Heterogeneous pipe joined with standard laser']
  Post-joining: ['pickling', 'passivation', 'grinding_polishing']

Freeze Dryers Shelf (ste

In [21]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IMPACT = "climate change: total (EF v3.1 - IPCC 2021)"
SCENARIO = "amount"


def get_value(record_group, impact=IMPACT, scenario=SCENARIO):
    """Extract an impact value from a {scenario: row} dict (unit-suffix tolerant)."""
    if not isinstance(record_group, dict):
        return 0.0

    row = record_group.get(scenario)
    if not isinstance(row, dict):
        return 0.0

    if impact in row:
        return row[impact]

    # Match with/without unit suffix, e.g. "name [kg CO2-Eq]" vs "name"
    base = impact.split(" [")[0]
    for key, value in row.items():
        if key.split(" [")[0] == base:
            return value
    return 0.0


# Indicators available in the results
INDICATORS = [
    key for key in joining_results[0]
    if key not in ("Database", "Process", "Key", "Scenario")
]


def build_records(impact):
    recs = []

    for product, res in assembled_results.items():
        pre = {
            f"{step} (pre)": get_value(record, impact)
            for step, record in res["pre_joining"].items()
        }
        post = {
            f"{step} (post)": get_value(record, impact)
            for step, record in res["post_joining"].items()
        }

        for tech_process, scenarios in res["joining"].items():
            tech = tech_process.replace(f"{product} joined with ", "")
            row = {
                "product": product,
                "label": tech,
                "joining": get_value(scenarios, impact),
            }
            row.update(pre)
            row.update(post)
            recs.append(row)

    return recs


# One colour per process step (colour follows the step, not its rank):
# pre-joining = blues, joining = orange, post-joining = violets.
# Steps that never co-occur for the same material share a shade.
STEP_COLORS = {
    "alkaline_degreasing (pre)":   "#2a78d6",
    "mechanical_treatment (pre)":  "#6fa3e6",
    "machining (pre)":             "#b3d0f2",
    "joining":                     "#eb6834",
    "chemical_etching (post)":     "#4b3bb5",
    "pickling (post)":             "#4b3bb5",
    "conversion_coating (post)":   "#8576e0",
    "passivation (post)":          "#8576e0",
    "grinding_polishing (post)":   "#c4bcf3",
}
STAGE_TITLES = {"pre": "Pre-joining", "join": "Joining", "post": "Post-joining"}
# Legend order: a header entry per stage (rank base), then its steps
STAGE_BASE = {"pre": 0, "join": 100, "post": 200}
STEP_ORDER = {k: STAGE_BASE[_s] + i + 1 for i, (k, _s) in enumerate(
    (k, "pre" if k.endswith("(pre)") else "post" if k.endswith("(post)") else "join")
    for k in STEP_COLORS)}


def _stage(col):
    if col.endswith("(pre)"):
        return "pre"
    if col.endswith("(post)"):
        return "post"
    return "join"


def _pretty(col):
    return col.replace(" (pre)", "").replace(" (post)", "").replace("_", " ").capitalize()


def plot_indicator(impact):
    plot_all = df(build_records(impact)).fillna(0.0)
    products = list(assembled_results.keys())
    unit = impact.split("[")[-1].rstrip("]") if "[" in impact else ""

    step_cols = [c for c in plot_all.columns if c not in ("product", "label")]
    ordered = (
        [c for c in step_cols if c.endswith("(pre)")]
        + (["joining"] if "joining" in step_cols else [])
        + [c for c in step_cols if c.endswith("(post)")]
    )

    n_cols = 3
    n_rows = -(-len(products) // n_cols)
    fig = make_subplots(
        rows=n_rows, cols=n_cols, subplot_titles=products,
        horizontal_spacing=0.07, vertical_spacing=0.16,
    )

    shown = set()
    for i, product in enumerate(products):
        r, c = divmod(i, n_cols)
        sub = plot_all[plot_all["product"] == product].set_index("label")[ordered]
        sub = sub.loc[:, (sub != 0).any(axis=0) | (sub.columns == "joining")]
        totals = sub.sum(axis=1)

        for col in sub.columns:
            stage = _stage(col)
            vals = sub[col]
            fig.add_trace(
                go.Bar(
                    x=sub.index, y=vals, name=_pretty(col),
                    marker=dict(color=STEP_COLORS.get(col, "#999999"),
                                line=dict(color="white", width=1.5)),
                    legendgroup=col,  # one group per step -> toggles it in every subplot
                    legendrank=STEP_ORDER.get(col, 999),
                    showlegend=col not in shown,
                    customdata=(vals / totals * 100).fillna(0),
                    hovertemplate=(
                        f"<b>%{{x}}</b><br>{_pretty(col)}: %{{y:.3g}} {unit}"
                        "<br>%{customdata:.1f}% of total<extra></extra>"
                    ),
                ),
                row=r + 1, col=c + 1,
            )
            shown.add(col)

        # Total label above each stack
        fig.add_trace(
            go.Scatter(
                x=totals.index, y=totals, mode="text",
                text=[f"{t:.3g}" for t in totals], textposition="top center",
                showlegend=False, hoverinfo="skip",
                textfont=dict(size=12),
            ),
            row=r + 1, col=c + 1,
        )
        fig.update_yaxes(
            title_text=unit if c == 0 else None, rangemode="tozero",
            range=[0, totals.max() * 1.15 if totals.max() > 0 else 1],
            row=r + 1, col=c + 1,
        )
        fig.update_xaxes(tickangle=-30, row=r + 1, col=c + 1)

    # Legend section headers: data-less traces that only show a bold stage title
    for stage in dict.fromkeys(_stage(c) for c in ordered):
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(size=0, opacity=0),
                name=f"<b>{STAGE_TITLES[stage]}</b>",
                legendgroup=f"header_{stage}", legendrank=STAGE_BASE[stage],
                hoverinfo="skip",
            ),
            row=1, col=1,
        )

    fig.update_layout(
        barmode="stack",
        template="plotly_white",
        height=380 * n_rows + 80,
        width=1200,
        title=dict(text=f"{impact}<br><sup>{SCENARIO} scenario - stacked contribution per process step</sup>",
                   x=0.01),
        legend=dict(groupclick="togglegroup", orientation="v", x=1.02, y=1,
                    tracegroupgap=12),
        margin=dict(t=100, r=40),
        hoverlabel=dict(bgcolor="white"),
        font=dict(size=12),
    )
    fig.update_annotations(font_size=14)  # subplot titles
    fig.show()


indicator_dropdown = widgets.Dropdown(
    options=INDICATORS,
    value=next((i for i in INDICATORS if i.startswith(IMPACT)), INDICATORS[0]),
    description="Indicator:",
    layout=widgets.Layout(width="600px"),
)
widgets.interact(plot_indicator, impact=indicator_dropdown)


interactive(children=(Dropdown(description='Indicator:', index=1, layout=Layout(width='600px'), options=('acid…

<function __main__.plot_indicator(impact)>